In [0]:
from pyspark.sql.functions import col
from delta.tables import DeltaTable

# Step 1: Read raw data from Bronze table
bronze_table = "workspace.ecommerce.bronze_events"
bronze_df = spark.read.table(bronze_table)

# Step 2: Data cleaning and validation
# Remove duplicates based on key columns
key_cols = ["event_time", "event_type", "product_id", "user_id", "user_session"]
silver_df = bronze_df.dropDuplicates(key_cols)

# Handle missing values (example: drop rows with null product_id or user_id)
silver_df = silver_df.dropna(subset=["product_id", "user_id"])

# Step 3: Write to Silver Delta table (overwrite for initial run)
silver_table = "workspace.ecommerce.silver_events"
silver_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)

# Step 4: Display sample Silver data
display(spark.read.table(silver_table).limit(10))
